# Unit Test Generator - Fine-tuning with Unsloth

This notebook fine-tunes a Qwen2.5-Coder model for generating Python unit tests, saves it to GGUF format, and runs inference using llama-cpp-python.

## 1. Load Base Model

In [1]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None  # Auto detection
load_in_4bit = True  # Use 4bit quantization to reduce memory

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-Coder-7B-Instruct-bnb-4bit",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/home/sachithra/miniconda3/envs/unsloth_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.12.5: Fast Qwen2 patching. Transformers: 4.57.3.
   \\   /|    NVIDIA GeForce RTX 3060. Num GPUs = 1. Max memory: 11.628 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.1+cu128. CUDA: 8.6. CUDA Toolkit: 12.8. Triton: 3.5.1
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


## 2. Add LoRA Adapters

In [2]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 8, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

Unsloth 2025.12.5 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


## 3. Dataset Exploration
Let's first take a quick look at the structures of the training and testing splits in `KAKA22/CodeRM-UnitTest` before preparing the prompt templates.

In [3]:
from datasets import load_dataset

# Load raw datasets for exploration
dataset_train_explore = load_dataset("KAKA22/CodeRM-UnitTest", split="train")
dataset_test_explore = load_dataset("KAKA22/CodeRM-UnitTest", split="test")

print(f"Train dataset size: {len(dataset_train_explore)} rows")
print(f"Test dataset size: {len(dataset_test_explore)} rows")
print("-" * 50)
print(f"Train columns: {list(dataset_train_explore.features.keys())}")
print(f"Test columns:  {list(dataset_test_explore.features.keys())}")
print("-" * 50)

# Display a sample from the dataset
sample = dataset_train_explore[0]
print("Example 'question' from train set:")
print(sample['question'][:300] + "..." if len(sample['question']) > 300 else sample['question'])


Train dataset size: 17562 rows
Test dataset size: 62900 rows
--------------------------------------------------
Train columns: ['task_id', 'question', 'code_ground_truth', 'code_generate', 'unit_tests']
Test columns:  ['task_id', 'question', 'code_ground_truth', 'code_generate', 'unit_tests']
--------------------------------------------------
Example 'question' from train set:
There are $n$ candy boxes in front of Tania. The boxes are arranged in a row from left to right, numbered from $1$ to $n$. The $i$-th box contains $r_i$ candies, candies have the color $c_i$ (the color can take one of three values ​​— red, green, or blue). All candies inside a single box have the sa...


## 4. Prepare Training Data

In [4]:
# Defined the prompt format for Unit Testing
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

EOS_TOKEN = tokenizer.eos_token

def formatting_prompts_func(examples):
    # The dataset uses these column names:
    # 'question': The problem description
    # 'code_ground_truth': The target code solution
    # 'unit_tests': A list of dicts, where each dict has a 'code' key with the actual test
    
    questions = examples["question"]
    codes     = examples["code_ground_truth"]
    unit_tests_lists = examples["unit_tests"]
    
    texts = []
    # Zip through the batch
    for question, code, tests_list in zip(questions, codes, unit_tests_lists):
        
        # Extract the actual python code for the tests from the list of dictionaries
        # We join multiple tests with newlines to create one full test suite
        if tests_list:
            # Handle cases where tests might be None or empty
            try:
                # The dataset structure has a 'code' key inside the unit_tests list items
                full_test_suite = "\n\n".join([t['code'] for t in tests_list if t.get('code')])
            except:
                full_test_suite = ""
        else:
            full_test_suite = ""
            
        # We combine the Question and the Code into the Input field
        # This gives the model context on WHAT the code is supposed to do
        complex_input = f"Problem Description:\n{question}\n\nCode to Test:\n{code}"
        
        instruction = "Write a comprehensive Python unit test suite for the provided code."
        
        text = alpaca_prompt.format(instruction, complex_input, full_test_suite) + EOS_TOKEN
        texts.append(text)
        
    return { "text" : texts, }

from datasets import load_dataset

# Load both the train and test splits of the dataset
train_dataset = load_dataset("KAKA22/CodeRM-UnitTest", split="train")
test_dataset = load_dataset("KAKA22/CodeRM-UnitTest", split="test")

#Shuffle and take a smaller subset of the test dataset for faster evaluation
test_dataset = test_dataset.shuffle(seed=42).select(range(5000))

# Apply the formatting function to both datasets
train_dataset = train_dataset.map(formatting_prompts_func, batched=True)
test_dataset = test_dataset.map(formatting_prompts_func, batched=True)

Map: 100%|██████████| 5000/5000 [00:04<00:00, 1024.51 examples/s]


## 5. Train the Model

In [5]:
from trl import SFTConfig, SFTTrainer

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    eval_dataset = test_dataset, # The correctly padded and formatted test dataset
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    args = SFTConfig(
        per_device_train_batch_size = 1,
        per_device_eval_batch_size = 1, # Make sure evaluations use appropriate batch size
        gradient_accumulation_steps = 8,
        # Use num_train_epochs = 1, warmup_ratio for full training runs!
        warmup_steps = 5,
        max_steps = 100,
        learning_rate = 2e-4,
        logging_steps = 10,
        eval_strategy = "steps", # Can evaluate dynamically over training duration
        eval_steps = 20, # Interval to evaluate the trained model against test_dataset split
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none", # Use TrackIO/WandB etc
    ),
)

[trl.trainer.sft_trainer|WARNING]You are using a per_device_train_batch_size of 1 with padding-free training. Using a batch size of 1 anihilate the benefits of padding-free training. Please consider increasing the batch size to at least 2.
Unsloth: Tokenizing ["text"] (num_proc=16): 100%|██████████| 5000/5000 [00:02<00:00, 2429.38 examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


In [6]:
trainer_stats = trainer.train()

The model is already on multiple devices. Skipping the move to device specified in `args`.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 17,562 | Num Epochs = 1 | Total steps = 100
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 20,185,088 of 7,635,801,600 (0.26% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,Validation Loss
20,0.869300,0.632146
40,0.788200,0.627727
60,0.834300,0.619600
80,0.753000,0.620042
100,0.753700,0.622829


Unsloth: Not an error, but Qwen2ForCausalLM does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient


## 6. Save Model as GGUF (Q8_0)

In [7]:
# Save model as GGUF Q8_0 format
output_dir = "gguf_model"

model.save_pretrained_gguf(output_dir, tokenizer, quantization_method="q8_0")
print(f"✅ Saved Q8_0 GGUF to '{output_dir}/'")
print(f"Model file: {output_dir}/unsloth.Q8_0.gguf")

Unsloth: Merging model weights to 16-bit format...
Found HuggingFace hub cache directory: /home/sachithra/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model-00001-of-00004.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files: 100%|██████████| 4/4 [56:37<00:00, 849.33s/it] 


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 4/4 [00:49<00:00, 12.37s/it]


Unsloth: Merge process complete. Saved to `/home/sachithra/Desktop/FYP/gguf_model`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF bf16 might take 3 minutes.
\        /    [2] Converting GGUF bf16 to ['q8_0'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Updating system package directories
Unsloth: All required system packages already installed!
Unsloth: Install llama.cpp and building - please wait 1 to 3 minutes
Unsloth: Cloning llama.cpp repository
Unsloth: Install GGUF and other packages


RuntimeError: Unsloth: GGUF conversion failed: === Unsloth: FAILED building llama.cpp ===
Make failed: [FAIL] Command `make clean` failed with exit code 2
stdout: Makefile:6: *** Build system changed:
 The Makefile build has been replaced by CMake.

 For build instructions see:
 https://github.com/ggml-org/llama.cpp/blob/master/docs/build.md

.  Stop.


CMake failed: [FAIL] Command `cmake . -B build -DBUILD_SHARED_LIBS=OFF -DGGML_CUDA=OFF -DLLAMA_CURL=ON` failed with error `LLAMA_CURL is deprecated and will be ignored`
Command is deprecated!
=== Full output log: ===
Cloning into 'llama.cpp'...
[2mUsing Python 3.11.14 environment at: /home/sachithra/miniconda3/envs/unsloth_env[0m
[2mResolved [1m27 packages[0m [2min 7.29s[0m[0m
[36m[1mDownloading[0m[39m pycountry [2m(7.7MiB)[0m
[36m[1mDownloading[0m[39m mistral-common [2m(6.2MiB)[0m
[36m[1mDownloading[0m[39m tiktoken [2m(1.1MiB)[0m
 [36m[1mDownloaded[0m[39m tiktoken
 [36m[1mDownloaded[0m[39m mistral-comm

## 7. Inference with GGUF Model

Now we can use the saved GGUF model with `llama-cpp-python` for inference.

In [ ]:
# Install llama-cpp-python if not already installed
# !pip install llama-cpp-python

from llama_cpp import Llama

# Load the GGUF model
llm = Llama(
    model_path="gguf_model/unsloth.Q8_0.gguf",
    n_ctx=2048,
    n_gpu_layers=-1  # Use all GPU layers (-1 = all, 0 = CPU only)
)

print("✅ GGUF model loaded successfully!")

In [ ]:
# Test code to generate unit tests for
user_code_input = """
def find_max(numbers):
    max_val = 0
    for num in numbers:
        if num > max_val:
            max_val = num
    return max_val
"""

user_problem_description = "A function designed to find the maximum integer in a list of numbers."

# Format the prompt
prompt = f"""Write a comprehensive Python unit test suite for the provided code.

Problem Description:
{user_problem_description}

Code to Test:
{user_code_input}"""

# Generate unit tests using the GGUF model
output = llm(
    prompt,
    max_tokens=512,
    temperature=0.7,
    top_p=0.9,
    echo=False
)

print("Generated Unit Tests:")
print("=" * 50)
print(output["choices"][0]["text"])